In [32]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [33]:
df=pd.read_csv('./dataGenerated/gurgaon_properties_post_feature_selection_v2.csv')

In [34]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,flat,sector 7,0.45,2,2,1,West,1000.0,0,Relatively New,Unfurnished,4.00,Low,Mid Floor
1,flat,sector 3,0.50,2,2,1,West,722.0,0,Old Property,Furnished,4.25,Low,Low Floor
2,flat,sohna road,0.40,2,2,3,East,661.0,0,New Property,Unfurnished,4.25,Low,High Floor
3,flat,sector 61,1.47,2,2,2,East,1333.0,0,Relatively New,Unfurnished,4.20,Low,Low Floor
4,flat,sector 92,0.70,2,2,3,East,1217.0,0,Under Construction,Unfurnished,4.00,Low,Mid Floor


In [35]:
# i am trying creating a linear model using all feature then try out to drop some feature and compare r2 score 

In [36]:
# Numerical = bedRoom, bathroom, built_up_area, servant room, combined_rating
# Ordinal = property_type, furnishing_type, luxury_category, balcony
# OHE = sector, age_category, facing

In [37]:
df['age_category'] = df['age_category'].replace(
    {
        'Relatively New':'new',
        'Moderately Old':'old',
        'New Property' : 'new',
        'Old Property' : 'old',
        'Under Construction' : 'under construction'
    }
)

In [38]:
df['property_type']=df['property_type'].replace({
    'flat':0,
    'house':1,
})

In [39]:
df['luxury_category'] = df['luxury_category'].replace({'Low':0,'Medium':1,'High':2})

In [40]:
df['furnishing_type']=df['furnishing_type'].replace({
    'Unfurnished':0,
    'Semi-Furnished':1,
    'Furnished':2,})

In [41]:
df['floor_category']=df['floor_category'].replace({
    'Low Floor':0,
    'Mid Floor':1,
    'High Floor':2,})

In [ ]:
df['balcony']=df['balcony'].replace({
    '0':0,
    '1':1,
    '2':2,
    '3':3,
    '3+':4})

In [43]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,0,sector 7,0.45,2,2,1,West,1000.0,0,new,0,4.00,0,1
1,0,sector 3,0.50,2,2,1,West,722.0,0,old,2,4.25,0,0
2,0,sohna road,0.40,2,2,3,East,661.0,0,new,0,4.25,0,2
3,0,sector 61,1.47,2,2,2,East,1333.0,0,new,0,4.20,0,0
4,0,sector 92,0.70,2,2,3,East,1217.0,0,under construction,0,4.00,0,1


In [44]:
df['balcony'].value_counts()

balcony
4    1122
3    1059
2     865
1     356
0     171
Name: count, dtype: int64

In [55]:
new_df_all = pd.get_dummies(df,columns=['sector','age_category','facing'],drop_first=True)

In [56]:
new_df_all.head()

,property_type,price,bedRoom,bathroom,balcony,built_up_area,servant room,furnishing_type,combined_rating,luxury_category,...,sector_sohna road road,age_category_old,age_category_under construction,facing_North,facing_North-East,facing_North-West,facing_South,facing_South-East,facing_South-West,facing_West
0,0,0.45,2,2,1,1000.0,0,0,4.00,0,...,False,False,False,False,False,False,False,False,False,True
1,0,0.50,2,2,1,722.0,0,2,4.25,0,...,False,True,False,False,False,False,False,False,False,True
2,0,0.40,2,2,3,661.0,0,0,4.25,0,...,False,False,False,False,False,False,False,False,False,False
3,0,1.47,2,2,2,1333.0,0,0,4.20,0,...,False,False,False,False,False,False,False,False,False,False
4,0,0.70,2,2,3,1217.0,0,0,4.00,0,...,False,False,True,False,False,False,False,False,False,False


In [58]:
X = new_df_all.drop(columns=['price'])
y = new_df_all['price']

In [59]:
y_log=np.log1p(y)

In [60]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [61]:
X_scaled=pd.DataFrame(X_scaled,columns=X.columns)

In [62]:
X_scaled

,property_type,bedRoom,bathroom,balcony,built_up_area,servant room,furnishing_type,combined_rating,luxury_category,floor_category,...,sector_sohna road road,age_category_old,age_category_under construction,facing_North,facing_North-East,facing_North-West,facing_South,facing_South-East,facing_South-West,facing_West
0,-0.521954,-0.826206,-0.822021,-1.509556,-0.706851,-0.749417,-0.700884,-0.940204,-0.810701,0.059122,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,3.483680
1,-0.521954,-0.826206,-0.822021,-1.509556,-0.926572,-0.749417,1.524134,-0.268324,-0.810701,-1.368187,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,3.483680
2,-0.521954,-0.826206,-0.822021,0.236525,-0.974784,-0.749417,-0.700884,-0.268324,-0.810701,1.486430,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3,-0.521954,-0.826206,-0.822021,-0.636516,-0.443660,-0.749417,-0.700884,-0.402700,-0.810701,-1.368187,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
4,-0.521954,-0.826206,-0.822021,0.236525,-0.535342,-0.749417,-0.700884,-0.940204,-0.810701,0.059122,...,-0.055571,-0.598795,5.954761,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3568,1.915878,-0.085977,-0.192287,0.236525,-0.153597,-0.749417,1.524134,-0.940204,1.233501,-1.368187,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,3.991698,-0.273105,-0.224198,-0.215096,-0.287053
3569,1.915878,0.654252,0.437448,0.236525,-0.074560,1.334371,-0.700884,1.747319,-0.810701,-1.368187,...,-0.055571,1.670021,-0.167933,-0.388710,1.846919,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3570,1.915878,-0.085977,-0.822021,0.236525,-0.430224,-0.749417,-0.700884,1.747319,-0.810701,-1.368187,...,-0.055571,1.670021,-0.167933,2.572613,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3571,1.915878,-0.085977,-0.192287,-0.636516,-0.430224,1.334371,-0.700884,1.747319,-0.810701,-1.368187,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053


In [63]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')

In [64]:
scores.mean(),scores.std()

(np.float64(0.8322961353567463), np.float64(0.023754983369239483))

In [65]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,furnishing_type,combined_rating,luxury_category,floor_category
0,0,sector 7,0.45,2,2,1,West,1000.0,0,new,0,4.00,0,1
1,0,sector 3,0.50,2,2,1,West,722.0,0,old,2,4.25,0,0
2,0,sohna road,0.40,2,2,3,East,661.0,0,new,0,4.25,0,2
3,0,sector 61,1.47,2,2,2,East,1333.0,0,new,0,4.20,0,0
4,0,sector 92,0.70,2,2,3,East,1217.0,0,under construction,0,4.00,0,1


In [85]:
new_df=df.drop(columns=['luxury_category','floor_category','furnishing_type'])
# 'facing','balcony','floor_category'

In [86]:
new_df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,facing,built_up_area,servant room,age_category,combined_rating
0,0,sector 7,0.45,2,2,1,West,1000.0,0,new,4.00
1,0,sector 3,0.50,2,2,1,West,722.0,0,old,4.25
2,0,sohna road,0.40,2,2,3,East,661.0,0,new,4.25
3,0,sector 61,1.47,2,2,2,East,1333.0,0,new,4.20
4,0,sector 92,0.70,2,2,3,East,1217.0,0,under construction,4.00


In [88]:
new_df_dropped=pd.get_dummies( new_df,columns=['sector','age_category','facing'],drop_first=True)

In [89]:
new_df_dropped.head()

,property_type,price,bedRoom,bathroom,balcony,built_up_area,servant room,combined_rating,sector_gwal pahari,sector_manesar,...,sector_sohna road road,age_category_old,age_category_under construction,facing_North,facing_North-East,facing_North-West,facing_South,facing_South-East,facing_South-West,facing_West
0,0,0.45,2,2,1,1000.0,0,4.00,False,False,...,False,False,False,False,False,False,False,False,False,True
1,0,0.50,2,2,1,722.0,0,4.25,False,False,...,False,True,False,False,False,False,False,False,False,True
2,0,0.40,2,2,3,661.0,0,4.25,False,False,...,False,False,False,False,False,False,False,False,False,False
3,0,1.47,2,2,2,1333.0,0,4.20,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,0.70,2,2,3,1217.0,0,4.00,False,False,...,False,False,True,False,False,False,False,False,False,False


In [90]:
X = new_df_dropped.drop(columns=['price'])
y = new_df_dropped['price']

In [91]:
y_log=np.log1p(y)

In [92]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [93]:
X_scaled = pd.DataFrame(X_scaled,columns=X.columns)

In [94]:
X_scaled

,property_type,bedRoom,bathroom,balcony,built_up_area,servant room,combined_rating,sector_gwal pahari,sector_manesar,sector_sector 1,...,sector_sohna road road,age_category_old,age_category_under construction,facing_North,facing_North-East,facing_North-West,facing_South,facing_South-East,facing_South-West,facing_West
0,-0.521954,-0.826206,-0.822021,-1.509556,-0.706851,-0.749417,-0.940204,-0.071157,-0.093553,-0.041013,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,3.483680
1,-0.521954,-0.826206,-0.822021,-1.509556,-0.926572,-0.749417,-0.268324,-0.071157,-0.093553,-0.041013,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,3.483680
2,-0.521954,-0.826206,-0.822021,0.236525,-0.974784,-0.749417,-0.268324,-0.071157,-0.093553,-0.041013,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3,-0.521954,-0.826206,-0.822021,-0.636516,-0.443660,-0.749417,-0.402700,-0.071157,-0.093553,-0.041013,...,-0.055571,-0.598795,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
4,-0.521954,-0.826206,-0.822021,0.236525,-0.535342,-0.749417,-0.940204,-0.071157,-0.093553,-0.041013,...,-0.055571,-0.598795,5.954761,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3568,1.915878,-0.085977,-0.192287,0.236525,-0.153597,-0.749417,-0.940204,-0.071157,-0.093553,-0.041013,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,3.991698,-0.273105,-0.224198,-0.215096,-0.287053
3569,1.915878,0.654252,0.437448,0.236525,-0.074560,1.334371,1.747319,-0.071157,-0.093553,-0.041013,...,-0.055571,1.670021,-0.167933,-0.388710,1.846919,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3570,1.915878,-0.085977,-0.822021,0.236525,-0.430224,-0.749417,1.747319,-0.071157,-0.093553,-0.041013,...,-0.055571,1.670021,-0.167933,2.572613,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053
3571,1.915878,-0.085977,-0.192287,-0.636516,-0.430224,1.334371,1.747319,-0.071157,-0.093553,-0.041013,...,-0.055571,1.670021,-0.167933,-0.388710,-0.541442,-0.250520,-0.273105,-0.224198,-0.215096,-0.287053


In [95]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X_scaled, y_log, cv=kfold, scoring='r2')
scores.mean(),scores.std()

(np.float64(0.8325960884589458), np.float64(0.02408614568664484))

In [96]:
lr = LinearRegression()

In [97]:
lr.fit(X_scaled,y_log)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [98]:
lr.coef_.shape

(117,)

In [99]:
coef_df = pd.DataFrame(lr.coef_.reshape(1,117),columns=X.columns).stack().reset_index().drop(columns=['level_0']).rename(columns={'level_1':'feature',0:'coef'})

In [102]:
coef_df.sort_values(by='coef',ascending=False)

,feature,coef
4,built_up_area,0.194858
0,property_type,0.129546
2,bathroom,0.068221
35,sector_sector 26,0.065359
73,sector_sector 65,0.062529
...,...,...
8,sector_manesar,-0.028295
102,sector_sector 92,-0.028433
104,sector_sector 95,-0.029199
48,sector_sector 4,-0.035567


In [106]:
y_log.std()

np.float64(0.5622161569622743)

In [113]:
X['built_up_area'].std()

np.float64(1265.41765843266)

In [114]:
0.194858 * (0.5622161569622743/1265.41765843266)

8.657403757826946e-05

In [115]:
8.657403757826946e-05 * 100

0.008657403757826946